<a href="https://colab.research.google.com/github/Koquda/intro_ai_ml/blob/main/assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NFL RAG assignment

## 1. Install libraries

Google Colab already includes PyTorch, NumPy, Transformers, and Sentence-Transformers, so only the missing libraries are installed. Do not reinstall `torch`: replacing Colab's build in a running session breaks `import torch` ("Only a single TORCH_LIBRARY can be used..."). If that error appears, use **Runtime → Restart session** and run the cells again from the top.

Standard-library modules such as `csv`, `json`, `pathlib`, and `typing` come with Python and need no installation. Select a GPU runtime in Google Colab if available.

In [20]:
%pip install -q faiss-cpu sentence-transformers accelerate sentencepiece

## 2. Import libraries

Keep all imports together: standard-library modules first, then third-party libraries. PyTorch must be imported before FAISS to avoid the OpenMP crash reproduced on macOS.

In [21]:
# Standard library
import csv
import io
import json
from pathlib import Path
from typing import Any, Literal, TypedDict
from urllib.request import urlopen
from zipfile import BadZipFile, ZipFile

# Third-party libraries (torch before faiss, see above)
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

## 3. Settings and data structures

Use [Nick Cantalupa's NFL Team Data 2003–2023](https://www.kaggle.com/datasets/nickcantalupa/nfl-team-data-2003-2023). Each CSV row becomes a document describing one team-season. `source` identifies the file and team; `date` stores the season year. MPNet embeddings and Qwen replace the supplied default models.

All tunable values (paths, models, chunk sizes, fixed replies) are defined once here, so the functions below contain no hidden numbers. The `TypedDict` classes document the shape of every record passed between steps.

In [22]:
# Data source
DATA_PATH = Path("documents/team_stats_2003_2023.csv")
DATA_URL = "https://www.kaggle.com/api/v1/datasets/download/nickcantalupa/nfl-team-data-2003-2023"
FIRST_SEASON, LAST_SEASON = 2003, 2023
REQUIRED_COLUMNS = {"team", "year", "wins", "losses", "points", "points_opp"}
MISSING_VALUES = {"", "na", "n/a", "nan", "null"}

# Models (both replace the defaults supplied with the assignment)
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"  # default: all-MiniLM-L6-v2
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # default: flan-t5-base

# Pipeline parameters
CHUNK_SIZE = 60  # words per chunk
CHUNK_OVERLAP = 15  # words shared by neighbouring chunks
TOP_K = 3  # team-season documents returned per search
MAX_NEW_TOKENS = 128  # generation limit for each model call
AGENT_ATTEMPTS = 2  # tries before rejecting invalid agent output

# Fixed replies that do not need the LLM
GREETING_REPLY = (
    f"Hello! Ask about an NFL team's regular-season statistics from {FIRST_SEASON} to {LAST_SEASON}."
)
UNSUPPORTED_REPLY = "I cannot answer from the dataset. It contains team regular-season statistics only."
NO_EVIDENCE_REPLY = "I cannot answer from the dataset because no matching evidence was found."

Action = Literal["retrieve", "greet", "unsupported"]
ACTIONS: tuple[Action, ...] = ("retrieve", "greet", "unsupported")


class Document(TypedDict):
    """One team-season record converted to searchable text.

    Attributes:
        id: Unique key in the form "<team>-<year>".
        text: Every statistic written as a "label: value." sentence.
        source: CSV file name and team, used as a metadata filter.
        date: Season year, used as a metadata filter.
        team: Full team name.
        statistics: The original CSV row, kept for manual fact-checking.
    """

    id: str
    text: str
    source: str
    date: str
    team: str
    statistics: dict[str, str]


class Chunk(TypedDict):
    """A searchable piece of a document that keeps its parent's metadata.

    Attributes:
        document_id: ID of the parent document.
        text: Chunk words, prefixed with the team and season.
        source: Copied from the parent document for filtering.
        date: Copied from the parent document for filtering.
    """

    document_id: str
    text: str
    source: str
    date: str


class Hit(Document):
    """A retrieved document plus its cosine similarity to the query."""

    score: float


class Decision(TypedDict):
    """The agent's choice for one message.

    Attributes:
        action: "retrieve" to search, "greet" for greetings, or "unsupported"
            for questions the dataset cannot answer.
        query: Search terms when retrieving, otherwise an empty string.
    """

    action: Action
    query: str


class Runtime(TypedDict):
    """Everything loaded once and shared by retrieval and generation.

    Attributes:
        documents: Documents keyed by ID, to return full parents for chunks.
        chunks: All chunks, in the same order as the embedding rows.
        embedder: Sentence-transformers model that encodes text.
        embeddings: Normalized chunk vectors, one row per chunk.
        index: FAISS index over all chunk embeddings.
        generator: Transformers text-generation pipeline for Qwen.
    """

    documents: dict[str, Document]
    chunks: list[Chunk]
    embedder: SentenceTransformer
    embeddings: np.ndarray
    index: faiss.IndexFlatIP
    generator: Any

## 4. Download the dataset

Download the CSV archive only when the local file is missing. If Kaggle downloading fails in Colab, upload `team_stats_2003_2023.csv` into `documents/`.

In [23]:
def download_dataset(path: Path) -> None:
    """Download the Kaggle archive and save the team-statistics CSV.

    Args:
        path: Where to save the CSV. Its file name selects the file inside the archive.

    Raises:
        RuntimeError: If the download fails or the archive lacks the CSV, so the
            user knows to place the file manually.
    """
    print("Downloading the NFL dataset from Kaggle...")
    try:
        with urlopen(DATA_URL, timeout=60) as response:
            archive_bytes = response.read()
        with ZipFile(io.BytesIO(archive_bytes)) as archive:
            matches = [name for name in archive.namelist() if Path(name).name == path.name]
            if len(matches) != 1:
                raise ValueError("The archive does not contain the expected CSV")
            csv_bytes = archive.read(matches[0])
    except (OSError, ValueError, BadZipFile) as error:
        raise RuntimeError(f"Download the Kaggle CSV manually and place it at {path}") from error

    path.parent.mkdir(parents=True, exist_ok=True)
    _ = path.write_bytes(csv_bytes)

## 5. Clean rows and create documents

Validate team and year fields, reject duplicate team-seasons, and convert statistics into labelled text. Keep missing statistics as unknown and preserve original CSV values.

The work is split into three small functions: `clean_value()` normalizes one cell, `row_to_document()` builds one document, and `clean_data()` validates the whole file.

In [24]:
def clean_value(value: str) -> str:
    """Strip whitespace and replace missing markers such as "NA" with "unknown"."""
    value = value.strip()
    return "unknown" if value.lower() in MISSING_VALUES else value


def row_to_document(row: dict[str, str], file_name: str) -> Document:
    """Turn one validated CSV row into a document with labelled statistics.

    Args:
        row: CSV row with non-empty "team" and "year" values.
        file_name: CSV file name, recorded in the document's source.

    Returns:
        A document whose text starts with the team and season, followed by
        one sentence per statistic, for example "points opp: 294."
    """
    team, year = row["team"].strip(), row["year"].strip()
    facts = [f"{team}, {year} NFL regular season."]
    for field, value in row.items():
        if field not in ("team", "year"):
            facts.append(f"{field.replace('_', ' ')}: {clean_value(value)}.")

    return Document(
        id=f"{team}-{year}",
        text=" ".join(facts),
        source=f"{file_name} / {team}",
        date=year,
        team=team,
        statistics=dict(row),
    )


def clean_data(path: Path) -> list[Document]:
    """Validate the CSV and convert every row into a team-season document.

    Args:
        path: CSV file with one row per team and season.

    Returns:
        One document per row, in file order.

    Raises:
        ValueError: If required columns are missing, a row is malformed, a team
            or year is invalid, or the same team-season appears twice.
    """
    documents: list[Document] = []
    seen_ids: set[str] = set()

    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        if not REQUIRED_COLUMNS.issubset(reader.fieldnames or []):
            raise ValueError("CSV is missing required team, year, record, or scoring columns")

        for row in reader:
            # DictReader stores extra cells under a None key and missing cells as None values.
            if None in row or None in row.values():
                raise ValueError("CSV contains a malformed row")

            team, year = row["team"].strip(), row["year"].strip()
            if not team or not year.isdecimal() or not FIRST_SEASON <= int(year) <= LAST_SEASON:
                raise ValueError(f"Invalid team or season year: {team!r}, {year!r}")

            document = row_to_document(row, path.name)
            if document["id"] in seen_ids:
                raise ValueError(f"Duplicate team-season: {document['id']}")
            seen_ids.add(document["id"])
            documents.append(document)

    return documents

## 6. Load the document collection

Reuse the local CSV or download it, then report how many team-season documents were loaded.

In [25]:
def load_documents(path: Path = DATA_PATH) -> list[Document]:
    """Load team-season documents, downloading the CSV first if it is missing.

    Args:
        path: Location of the CSV file.

    Returns:
        The cleaned documents.

    Raises:
        ValueError: If the CSV contains no data rows.
    """
    if not path.exists():
        download_dataset(path)

    documents = clean_data(path)
    if not documents:
        raise ValueError("The CSV is empty")

    print(f"Loaded {len(documents)} team-season documents.")
    return documents

## 7. Split text into chunks

Use the supplied example's 60-word chunks and 15-word overlap to retain context at chunk boundaries.

In [26]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into word-bounded chunks that overlap by a fixed number of words.

    Example:
        chunk_text("one two three four five six", chunk_size=4, overlap=2)
        returns ["one two three four", "three four five six"].

    Args:
        text: Text to split.
        chunk_size: Maximum number of words per chunk.
        overlap: Number of words repeated at the start of the next chunk.

    Returns:
        The chunks in order, or an empty list for empty text.

    Raises:
        ValueError: If overlap is negative or not smaller than chunk_size.
    """
    if not 0 <= overlap < chunk_size:
        raise ValueError("Overlap must be nonnegative and smaller than chunk size")

    words = text.split()
    step = chunk_size - overlap
    pieces: list[str] = []
    for start in range(0, len(words), step):
        end = start + chunk_size
        pieces.append(" ".join(words[start:end]))
        if end >= len(words):
            break  # the last chunk already reaches the end of the text
    return pieces

## 8. Attach metadata to chunks

Preserve the parent document ID, source, and season year on each searchable chunk. Each chunk is also prefixed with its team and season, so a chunk from the middle of a document still matches queries that name the team or year.

In [27]:
def chunk_documents(
    documents: list[Document], size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP,
) -> list[Chunk]:
    """Chunk every document and copy its metadata onto each chunk.

    Args:
        documents: Documents to split.
        size: Maximum number of words per chunk.
        overlap: Number of words shared by neighbouring chunks.

    Returns:
        All chunks, grouped by document in input order.
    """
    chunks: list[Chunk] = []
    for document in documents:
        label = f"{document['team']}, {document['date']}."
        for piece in chunk_text(document["text"], size, overlap):
            chunks.append(Chunk(
                document_id=document["id"],
                text=f"{label} {piece}",
                source=document["source"],
                date=document["date"],
            ))
    return chunks

## 9. Define model loading and indexing

Normalize chunk embeddings and add them to FAISS `IndexFlatIP` for cosine similarity. Load the Qwen generator once. Qwen uses `text-generation` because it is a decoder model.

`build_index()` is shared with retrieval, which builds a smaller index when metadata filters are applied.

In [28]:
def build_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """Create a FAISS inner-product index containing the given vectors.

    The vectors must be normalized, so the inner product equals cosine similarity.
    """
    vectors = np.ascontiguousarray(embeddings, dtype="float32")
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def build_runtime(documents: list[Document]) -> Runtime:
    """Chunk and embed the documents, build the FAISS index, and load the LLM.

    Runs once because loading models and embedding every chunk is slow.

    Args:
        documents: Documents to make searchable.

    Returns:
        The runtime shared by retrieval and generation.
    """
    chunks = chunk_documents(documents)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading embedding model {EMBEDDING_MODEL}...", flush=True)
    embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)

    print(f"Embedding {len(chunks)} chunks...", flush=True)
    embeddings = embedder.encode(
        [chunk["text"] for chunk in chunks], normalize_embeddings=True, show_progress_bar=True,
    )
    embeddings = np.asarray(embeddings, dtype="float32")
    index = build_index(embeddings)

    print(f"Indexed {index.ntotal} chunks. Loading {LLM_MODEL} on {device}...", flush=True)
    generator = pipeline("text-generation", model=LLM_MODEL, device=device)

    return Runtime(
        documents={document["id"]: document for document in documents},
        chunks=chunks,
        embedder=embedder,
        embeddings=embeddings,
        index=index,
        generator=generator,
    )

## 10. Define filtered retrieval

Apply source and date filters before FAISS ranks chunks. Map filtered positions back to the correct documents, and return full parent text for context.

Retrieval runs in three steps: filter chunks by metadata, rank the remaining chunks with FAISS, then keep each parent document once.

In [29]:
def matching_positions(chunks: list[Chunk], source: str = "", year: str = "") -> list[int]:
    """Return the positions of chunks whose metadata matches the filters.

    An empty filter matches every chunk.
    """
    return [
        position for position, chunk in enumerate(chunks)
        if (not source or chunk["source"] == source) and (not year or chunk["date"] == year)
    ]


def retrieve(
    runtime: Runtime, query: str, source: str = "", year: str = "", k: int = TOP_K,
) -> list[Hit]:
    """Find the documents most similar to the query, after metadata filtering.

    Args:
        runtime: Loaded models, chunks, and index.
        query: Search text written by the agent.
        source: Optional exact source, for example "team_stats_2003_2023.csv / Kansas City Chiefs".
        year: Optional season year, for example "2023".
        k: Maximum number of documents to return.

    Returns:
        Up to k full parent documents, best cosine similarity first. Empty if
        no chunk matches the filters.

    Raises:
        ValueError: If the query is blank or k is less than 1.
    """
    if not query.strip() or k < 1:
        raise ValueError("Enter a query and a positive number of results")

    # 1. Filter by metadata before any ranking.
    positions = matching_positions(runtime["chunks"], source, year)
    if not positions:
        return []

    # 2. Rank with the full index, or with a temporary index of the filtered chunks.
    if source or year:
        index = build_index(runtime["embeddings"][positions])
    else:
        index = runtime["index"]
    query_vector = runtime["embedder"].encode([query], normalize_embeddings=True)
    scores, local_positions = index.search(np.asarray(query_vector, dtype="float32"), len(positions))

    # 3. Several chunks can share a parent, so keep each document's best-scoring chunk only.
    hits: list[Hit] = []
    seen_ids: set[str] = set()
    for score, local_position in zip(scores[0], local_positions[0]):
        document_id = runtime["chunks"][positions[int(local_position)]]["document_id"]
        if document_id in seen_ids:
            continue
        seen_ids.add(document_id)
        hits.append(Hit(**runtime["documents"][document_id], score=float(score)))
        if len(hits) == k:
            break
    return hits

## 11. Define text generation

Use the Transformers chat pipeline and return only newly generated text.

In [30]:
def ask_model(runtime: Runtime, instruction: str, prompt: str) -> str:
    """Send a system instruction and user prompt to the LLM and return its reply.

    Decoding is greedy (no sampling), so the same prompt gives the same answer.

    Args:
        runtime: Runtime holding the text-generation pipeline.
        instruction: System message describing the model's role.
        prompt: User message.

    Returns:
        Only the newly generated text, without the prompt.
    """
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": prompt},
    ]
    output = runtime["generator"](
        messages, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, return_full_text=False,
    )
    return output[0]["generated_text"].strip()

## 12. Define the retrieval agent

The LLM chooses retrieval, greeting, or unsupported. When retrieving, it writes the search query. Few-shot examples show the expected structured decisions.

`parse_decision()` checks the model's JSON. If the reply is invalid, the agent shows the model its mistake and tries once more before giving up.

In [31]:
AGENT_INSTRUCTION = (
    'Classify the message and return only JSON with action and query. '
    f'The only available data is NFL TEAM regular-season statistics for {FIRST_SEASON}-{LAST_SEASON}. '
    'action="unsupported" for individual PLAYER statistics, playoff results, or predictions. '
    'action="greet" for greetings or help. action="retrieve" for team-statistics questions. '
    'For retrieve, write search terms preserving teams, seasons, and metrics. '
    'For greet or unsupported, query must be empty.'
)

# Few-shot examples follow the prompting approach covered in the RAG slides.
AGENT_EXAMPLES = (
    'Message: Hello!\nJSON: {"action": "greet", "query": ""}\n'
    'Message: How many games did Dallas win in 2020?\n'
    'JSON: {"action": "retrieve", "query": "Dallas Cowboys 2020 wins"}\n'
    'Message: How many touchdowns did an individual quarterback throw?\n'
    'JSON: {"action": "unsupported", "query": ""}\n'
)


def parse_decision(raw: str) -> Decision | None:
    """Parse the agent's reply, returning None if it is not a valid decision.

    A valid reply is a JSON object with exactly the keys "action" and "query",
    a known action, and a non-empty query when the action is "retrieve". The
    query is cleared for other actions because they never search.
    """
    try:
        reply = json.loads(raw)
    except ValueError:
        return None

    if not isinstance(reply, dict) or set(reply) != {"action", "query"}:
        return None
    action, query = reply["action"], reply["query"]
    if action not in ACTIONS or not isinstance(query, str):
        return None
    if action == "retrieve" and not query.strip():
        return None
    return Decision(action=action, query=query if action == "retrieve" else "")


def agent_decision(runtime: Runtime, question: str) -> Decision:
    """Let the LLM decide whether the question needs retrieval, and with what query.

    Args:
        runtime: Runtime holding the LLM.
        question: The user's message.

    Returns:
        The validated decision.

    Raises:
        ValueError: If every attempt returns invalid output.
    """
    prompt = f"{AGENT_EXAMPLES}Message: {question}\nJSON:"
    for _ in range(AGENT_ATTEMPTS):
        raw = ask_model(runtime, AGENT_INSTRUCTION, prompt)
        decision = parse_decision(raw)
        if decision is not None:
            return decision
        # Show the model its invalid output and ask again.
        prompt = f"{AGENT_EXAMPLES}Message: {question}\nInvalid output: {raw}\nReturn valid JSON:"
    raise ValueError(f"The agent did not produce a valid decision after {AGENT_ATTEMPTS} attempts")

## 13. Build the grounded prompt

Combine the question with numbered evidence. Request answers supported by the context and an explicit refusal when the required information is missing.

In [32]:
ANSWER_RULES = (
    "Answer using ONLY the context below. Cite facts using [1], [2], or [3]. "
    "If the answer is missing, say 'I cannot answer from the dataset.' "
    "Every record is a TEAM total. No individual player or playoff records are provided. "
    "Never attribute a team's passing or rushing totals to an individual player. "
    "Missing values are unknown. Points means points scored; points opp means points allowed.\n"
    "Example context: [1] Example Team, 2020. wins: 8. pass td: 20.\n"
    "Example question: How many wins did Example Team have?\nExample answer: 8 wins [1].\n"
    "Example question: How many touchdowns did an individual player throw?\n"
    "Example answer: I cannot answer from the dataset.\n\n"
)


def build_prompt(query: str, retrieved_chunks: list[Hit]) -> str:
    """Combine the answering rules, numbered evidence, and the question.

    Args:
        query: The user's original question.
        retrieved_chunks: Retrieved documents, numbered [1], [2], ... for citations.

    Returns:
        The full prompt for the LLM.
    """
    context = "\n\n".join(
        f"[{number}] {hit['text']}" for number, hit in enumerate(retrieved_chunks, start=1)
    )
    return f"{ANSWER_RULES}Context:\n{context}\n\nQuestion: {query}\nAnswer:"

## 14. Define answer generation

Handle an empty retrieval result directly; otherwise send the grounded prompt to the LLM.

In [33]:
ANSWER_INSTRUCTION = "Use only the supplied evidence. Team totals cannot answer individual player questions."


def generate_answer(runtime: Runtime, query: str, retrieved_chunks: list[Hit]) -> str:
    """Answer the question from the retrieved evidence.

    Args:
        runtime: Runtime holding the LLM.
        query: The user's original question.
        retrieved_chunks: Evidence from retrieve().

    Returns:
        The model's answer, or a fixed refusal without calling the model when
        there is no evidence.
    """
    if not retrieved_chunks:
        return NO_EVIDENCE_REPLY
    return ask_model(runtime, ANSWER_INSTRUCTION, build_prompt(query, retrieved_chunks))

## 15. Connect the RAG pipeline

Run the agent, retrieve when requested, and display sources, metadata, relevance scores, and the answer.

In [34]:
def show_hits(hits: list[Hit]) -> None:
    """Print each retrieved document with its metadata and similarity score."""
    for number, hit in enumerate(hits, start=1):
        print(f"\n[{number}] {hit['id']}, source={hit['source']}, date={hit['date']}, cosine={hit['score']:.3f}")
        print(hit["text"])


def rag_pipeline(runtime: Runtime, question: str, source: str = "", year: str = "") -> str:
    """Answer a question with the full agentic RAG pipeline.

    The agent first decides what to do. Greetings and unsupported questions get
    a fixed reply; only team-statistics questions trigger retrieval and generation.

    Args:
        runtime: Loaded models, chunks, and index.
        question: The user's question.
        source: Optional source filter applied before ranking.
        year: Optional season-year filter applied before ranking.

    Returns:
        The answer text.
    """
    print("Agent is deciding whether to retrieve...", flush=True)
    decision = agent_decision(runtime, question)
    print("Agent decision:", decision, "Filters:", {"source": source, "date": year})

    if decision["action"] == "greet":
        return GREETING_REPLY
    if decision["action"] == "unsupported":
        return UNSUPPORTED_REPLY

    hits = retrieve(runtime, decision["query"], source, year)
    show_hits(hits)
    print("Generating answer...", flush=True)
    return generate_answer(runtime, question, hits)

## 16. Define demonstration questions

Test source/date filtering, different seasons, a comparison, unsupported questions, and a greeting. Print the original CSV records so numerical answers can be checked manually.

In [35]:
def evaluate(runtime: Runtime, documents: list[Document]) -> None:
    """Run the demonstration questions and print the CSV records for checking.

    The examples cover source and date filters, a single filter, two seasons,
    an individual-player question, a missing season, and a greeting.

    Args:
        runtime: Loaded models, chunks, and index.
        documents: All documents, used to find the Kansas City Chiefs records.

    Raises:
        ValueError: If the dataset has no Kansas City Chiefs records.
    """
    chiefs = [document for document in documents if document["team"] == "Kansas City Chiefs"]
    if not chiefs:
        raise ValueError("Examples require Kansas City Chiefs records")
    chiefs_source = chiefs[0]["source"]

    # (question, source filter, year filter); empty strings mean no filter.
    examples = [
        ("How many wins and points did the Kansas City Chiefs have in 2023?", chiefs_source, "2023"),
        ("How many wins and points did the Kansas City Chiefs have in 2022?", chiefs_source, "2022"),
        ("What were Kansas City's rushing yards in 2023?", "", "2023"),
        ("How many wins did the Kansas City Chiefs have in 2022 and 2023?", chiefs_source, ""),
        ("How many passing touchdowns did Patrick Mahomes individually throw in 2023?", chiefs_source, "2023"),
        ("How many wins did the Kansas City Chiefs have in 2025?", chiefs_source, "2025"),
        ("Hello!", "", ""),
    ]
    for question, source, year in examples:
        print("\nQuestion:", question)
        try:
            print("Answer:", rag_pipeline(runtime, question, source, year))
        except ValueError as error:
            print("Evaluation failure:", error)

    print("\nOriginal CSV records for checking factual answers:")
    for document in chiefs:
        if document["date"] in ("2022", "2023"):
            print(document["date"], document["statistics"])
    print("Check numbers, citations, refusals, and agent decisions against the displayed evidence.")

## 17. Load data and models

The first run downloads model files if necessary. Later cells reuse the same loaded resources.

In [36]:
documents = load_documents()
runtime = build_runtime(documents)

Loaded 672 team-season documents.
Loading embedding model sentence-transformers/all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding 2016 chunks...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Indexed 2016 chunks. Loading Qwen/Qwen2.5-1.5B-Instruct on cpu...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## 18. Run and inspect the examples

Save the displayed agent decisions, retrieved context, answers, and original CSV values as notebook outputs. These are regular-season team totals, not player or playoff statistics. Check citations and numerical answers, particularly for multi-season comparisons. Similarity scores measure relevance, not confidence.

In [37]:
evaluate(runtime, documents)


Question: How many wins and points did the Kansas City Chiefs have in 2023?
Agent is deciding whether to retrieve...
Agent decision: {'action': 'retrieve', 'query': 'Kansas City Chiefs 2023 wins'} Filters: {'source': 'team_stats_2003_2023.csv / Kansas City Chiefs', 'date': '2023'}

[1] Kansas City Chiefs-2023, source=team_stats_2003_2023.csv / Kansas City Chiefs, date=2023, cosine=0.790
Kansas City Chiefs, 2023 NFL regular season. wins: 11. losses: 6. win loss perc: 0.647. points: 371. points opp: 294. points diff: 77. mov: 4.5. g: 17. total yards: 5972. plays offense: 1080. yds per play offense: 5.5. turnovers: 28. fumbles lost: 11. first down: 350. pass cmp: 421. pass att: 635. pass yds: 4188. pass td: 28. pass int: 17. pass net yds per att: 6.3. pass fd: 216. rush att: 417. rush yds: 1784. rush td: 9. rush yds per att: 4.3. rush fd: 107. penalties: 96. penalties yds: 845. pen fd: 27. score pct: 39.3. turnover pct: 14.6. exp pts tot: 68.59. ties: unknown.
Generating answer...
Answer

## 19. Ask your own question

Run this optional cell and enter one question, including any team and season. Rerun the cell to ask another question. Automated verification skips it because it waits for input.

In [38]:
question = input("Question: ").strip()
if question:
    print("Answer:", rag_pipeline(runtime, question))

Question: 49ers regular season in 2023
Agent is deciding whether to retrieve...
Agent decision: {'action': 'unsupported', 'query': ''} Filters: {'source': '', 'date': ''}
Answer: I cannot answer from the dataset. It contains team regular-season statistics only.
